# Document processing pipeline: notebook

This notebook runs the same pipeline as the BullMQ worker (`src/workers/document-processor.worker.ts` in the `doc-processing` repo), one step per cell, with no Redis or queue involved. It imports the real modules from that repo through the `@doc/` alias in `deno.json`, so what you see here is what production does.

| Step | Module | Worker progress |
|---|---|---|
| 1. Load bytes (local file or S3) | `storage/s3.service` | 15% |
| 2. Validate magic bytes, size, SHA-256 | `validation/file-validator` | 35% |
| 3. Triage + extract the canonical AST | `extraction/triage.service` | 60% |
| 4. Render Markdown | `generation/markdown-renderer` | 80% |
| 5. Persist AST + Markdown (local, optionally S3) | `storage/s3.service` | 100% |

**One-time setup**
1. Clone this repo next to `doc-processing` (or change the paths in `deno.json`), then run `npm install` in `doc-processing`.
2. Install [Deno](https://deno.com) 2.x and register the kernel: `deno jupyter --install`
3. Open this notebook and pick the **Deno** kernel.

**Keys**: digital PDFs and Word files need no credentials. Scanned PDFs and images go through Gemini (`GEMINI_API_KEY`, `GEMINI_MODEL`), and S3 input/output needs the `AWS_*` vars. The settings cell loads them from the main repo's `.env`. You'll see an "Invalid environment configuration" warning on import if that file is incomplete; that's fine for local digital files.

## 0. Settings

In [ ]:
// ── What to process ────────────────────────────────────────────────────────
const INPUT_PATH = 'samples/sample.pdf';      // local file (relative to repo root)
const S3_KEY: string | null = null;           // set to an S3 key to download from the bucket instead
const TENANT_ID: string | undefined = undefined;

// ── Where results go ───────────────────────────────────────────────────────
const OUTPUT_DIR = 'notebook-output';         // local copy of every run (gitignored)
const UPLOAD_TO_S3 = false;                   // true = also upload AST + Markdown like the worker does

// ── Logging ────────────────────────────────────────────────────────────────
const LOG_LEVEL = 'debug';                    // trace | debug | info | warn | error

// Everything below must happen before src/config is imported.
import dotenv from 'dotenv';
import nodePath from 'node:path';

// The doc-processing repo root, taken from the @doc/ alias in deno.json
const denoConfig = JSON.parse(await Deno.readTextFile('deno.json'));
const MAIN_REPO = nodePath.resolve(denoConfig.imports['@doc/'], '..');
dotenv.config({ path: nodePath.join(MAIN_REPO, '.env'), quiet: true });

// 'test' disables the pino-pretty worker-thread transport, whose output would not show up in cells.
Deno.env.set('NODE_ENV', 'test');
Deno.env.set('LOG_LEVEL', LOG_LEVEL);
console.log(`Using doc-processing at ${MAIN_REPO}`);

## Imports and notebook logger

The pipeline logs through `log()` from `src/shared/context.ts`, which picks up the logger in the active `AsyncLocalStorage` context. `withContext()` wraps each step in a context whose logger prints straight into the cell.

In [ ]:
import { Buffer } from 'node:buffer';
import process from 'node:process';
import path from 'node:path';
import pino from 'pino';

import { requestContext } from '@doc/shared/context.ts';
import { AppError } from '@doc/shared/errors.ts';
import { startTimer } from '@doc/shared/timer.ts';
import { validateBuffer, type ValidatedFile } from '@doc/modules/validation/file-validator.ts';
import { triageAndExtract, type ExtractionTriageResult } from '@doc/modules/extraction/triage.service.ts';
import { renderCanonicalToMarkdown } from '@doc/modules/generation/markdown-renderer.ts';
import { renderCanonicalToHtml } from '@doc/modules/generation/html-renderer.ts';
import { downloadBufferFromS3, uploadBufferToS3 } from '@doc/modules/storage/s3.service.ts';
import type { CanonicalDocument } from '@doc/modules/canonical/canonical.types.ts';

const LEVEL_NAMES: Record<number, string> = { 10: 'TRACE', 20: 'DEBUG', 30: 'INFO', 40: 'WARN', 50: 'ERROR', 60: 'FATAL' };

const nbLogger = pino(
  { level: LOG_LEVEL, base: undefined, timestamp: false },
  {
    write(line: string) {
      const { level, msg, ...fields } = JSON.parse(line);
      const extra = Object.keys(fields).length ? '  ' + JSON.stringify(fields) : '';
      console.log(`[${LEVEL_NAMES[level] ?? level}] ${msg}${extra}`);
    },
  },
);

let runCounter = 0;
function withContext<T>(fn: () => Promise<T>): Promise<T> {
  return requestContext.run(
    { requestId: `nb-${++runCounter}`, logger: nbLogger, startTime: process.hrtime.bigint() },
    fn,
  );
}

console.log('Pipeline modules loaded');

## 1. Load the file

In [ ]:
const originalName = S3_KEY ? path.basename(S3_KEY) : path.basename(INPUT_PATH);
const fileBuffer: Buffer = S3_KEY
  ? await withContext(() => downloadBufferFromS3(S3_KEY))
  : Buffer.from(await Deno.readFile(INPUT_PATH));

console.table({
  source: S3_KEY ? `s3://${S3_KEY}` : INPUT_PATH,
  originalName,
  sizeKb: +(fileBuffer.length / 1024).toFixed(1),
  headerBytes: fileBuffer.subarray(0, 8).toString('hex').match(/../g)!.join(' '),
});

## 2. Validate

Magic-byte MIME detection, the `MAX_FILE_SIZE_MB` limit, and the SHA-256 hash. Unsupported or empty files throw an `AppError` here.

In [ ]:
const validated: ValidatedFile = await withContext(() => validateBuffer(fileBuffer, originalName));

const { buffer: _buffer, ...validatedSummary } = validated;
console.table(validatedSummary);

## 3. Triage and extract the canonical AST

`triageAndExtract` routes by MIME type:
- **PDF**: tries the digital text layer (pdfjs). If any page is scanned or sparse, the whole document goes to Gemini VLM OCR.
- **DOCX / DOC**: mammoth / word-extractor, each falling back to the other.
- **Images**: Gemini VLM OCR.

In [ ]:
const stopExtractTimer = startTimer();
const triage: ExtractionTriageResult = await withContext(() => triageAndExtract(validated));
const extractMs = stopExtractTimer();
const ast: CanonicalDocument = triage.ast;

const blockCounts: Record<string, number> = {};
for (const section of ast.sections) {
  for (const block of section.blocks) blockCounts[block.type] = (blockCounts[block.type] ?? 0) + 1;
}

console.table({
  documentId: ast.id,
  title: ast.title,
  route: triage.routeTaken,
  pages: triage.pageCount ?? '-',
  sections: ast.sections.length,
  citations: ast.citations.length,
  extractMs,
});
console.table(blockCounts);

### Inspect the AST

An outline of every block. Set `SHOW_FULL_JSON = true` to print the whole document instead.

In [ ]:
const SHOW_FULL_JSON = false;

function blockPreview(block: CanonicalDocument['sections'][number]['blocks'][number]): string {
  const clip = (s: string) => (s.length > 90 ? s.slice(0, 87) + '...' : s);
  switch (block.type) {
    case 'heading': return `H${block.level}  ${clip(block.text)}`;
    case 'paragraph':
    case 'callout': return clip(block.runs.map((r) => r.text).join(''));
    case 'list': return `${block.items.length} ${block.ordered ? 'ordered' : 'bulleted'} items`;
    case 'table': return `${block.headerRows.length} header + ${block.bodyRows.length} body rows`;
    case 'page_break': return '';
  }
}

if (SHOW_FULL_JSON) {
  console.log(JSON.stringify(ast, null, 2));
} else {
  for (const section of ast.sections) {
    console.log(`§ ${section.title ?? '(untitled section)'}  [${section.id}]`);
    for (const block of section.blocks) console.log(`   ${block.type.padEnd(10)} ${blockPreview(block)}`);
  }
}

## 4. Render Markdown

In [ ]:
const markdownText = renderCanonicalToMarkdown(ast);
console.log(`${markdownText.length} chars, ${markdownText.split('\n').length} lines`);

// Rendered preview (the raw text is written to disk in step 5)
Deno.jupyter.md`${markdownText}`

### Optional: HTML preview

The same AST through the HTML renderer used for PDF generation, shown in an iframe so its styles don't leak into the notebook.

In [ ]:
const html = renderCanonicalToHtml(ast);
const srcdoc = html.replaceAll('&', '&amp;').replaceAll('"', '&quot;');
Deno.jupyter.html`<iframe srcdoc="${srcdoc}" style="width:100%;height:520px;border:1px solid #ccc;background:#fff"></iframe>`

## 5. Persist outputs

Always writes a local copy to `OUTPUT_DIR/<documentId>/`. With `UPLOAD_TO_S3 = true` it also uploads to the same keys the worker uses.

In [ ]:
const astJson = JSON.stringify(ast, null, 2);

const localDir = path.join(OUTPUT_DIR, ast.id);
await Deno.mkdir(localDir, { recursive: true });
await Deno.writeTextFile(path.join(localDir, 'canonical_ast.json'), astJson);
await Deno.writeTextFile(path.join(localDir, 'document.md'), markdownText);
console.log(`Saved locally → ${localDir}`);

if (UPLOAD_TO_S3) {
  const tenantPrefix = TENANT_ID ? `orgs/${TENANT_ID}/` : '';
  const baseKey = `uploads/${tenantPrefix}docs/${ast.id}`;
  const [astUpload, mdUpload] = await withContext(() =>
    Promise.all([
      uploadBufferToS3(Buffer.from(astJson, 'utf-8'), 'canonical_ast.json', {
        key: `${baseKey}/canonical_ast.json`,
        contentType: 'application/json',
        tenantId: TENANT_ID,
        docId: ast.id,
      }),
      uploadBufferToS3(Buffer.from(markdownText, 'utf-8'), 'document.md', {
        key: `${baseKey}/document.md`,
        contentType: 'text/markdown; charset=utf-8',
        tenantId: TENANT_ID,
        docId: ast.id,
      }),
    ])
  );
  console.table({ ast: astUpload.key, markdown: mdUpload.key });
} else {
  console.log('S3 upload skipped (UPLOAD_TO_S3 = false)');
}

## Batch: whole pipeline over a folder

Steps 1–4 chained into one function and run over every file in `BATCH_DIR`. Failures are recorded per file instead of stopping the loop. Outputs are written locally like step 5 (no S3).

In [ ]:
const BATCH_DIR = 'samples';

interface BatchRow {
  file: string;
  status: 'ok' | 'failed';
  route?: string;
  pages?: number | string;
  blocks?: number;
  mdChars?: number;
  ms: number;
  error?: string;
}

async function processDocument(filePath: string): Promise<BatchRow> {
  const stop = startTimer();
  const file = path.basename(filePath);
  try {
    const buffer = Buffer.from(await Deno.readFile(filePath));
    const result = await withContext(async () => {
      const v = await validateBuffer(buffer, file);
      return triageAndExtract(v);
    });
    const md = renderCanonicalToMarkdown(result.ast);
    const dir = path.join(OUTPUT_DIR, result.ast.id);
    await Deno.mkdir(dir, { recursive: true });
    await Deno.writeTextFile(path.join(dir, 'canonical_ast.json'), JSON.stringify(result.ast, null, 2));
    await Deno.writeTextFile(path.join(dir, 'document.md'), md);
    return {
      file,
      status: 'ok',
      route: result.routeTaken,
      pages: result.pageCount ?? '-',
      blocks: result.ast.sections.reduce((n, s) => n + s.blocks.length, 0),
      mdChars: md.length,
      ms: stop(),
    };
  } catch (err) {
    const error = err instanceof AppError ? `${err.code}: ${err.message}` : String(err);
    return { file, status: 'failed', ms: stop(), error };
  }
}

const rows: BatchRow[] = [];
for await (const entry of Deno.readDir(BATCH_DIR)) {
  if (entry.isFile) rows.push(await processDocument(path.join(BATCH_DIR, entry.name)));
}
console.table(rows);